In [4]:
from sklearn.model_selection import train_test_split

X = df_fraud.drop("Class", axis=1)
y = df_fraud["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("\nFraud count in Train set:", y_train.sum())
print("Fraud count in Test set:", y_test.sum())

Training set shape: (8000, 10)
Testing set shape: (2000, 10)

Fraud count in Train set: 44
Fraud count in Test set: 11


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

lr_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "classifier__C": [0.01, 0.1, 1.0]
}

lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

lr_grid.fit(X_train, y_train)

print("Best Logistic Regression Parameters:", lr_grid.best_params_)
print("Best Cross-Validation ROC-AUC:", round(lr_grid.best_score_, 4))

Best Logistic Regression Parameters: {'classifier__C': 0.1, 'smote__k_neighbors': 7}
Best Cross-Validation ROC-AUC: 0.9602


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("smote", SMOTE(random_state=42)),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_param_grid = {
    "smote__k_neighbors": [3, 5, 7],
    "classifier__max_depth": [10, 20, None]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest Parameters:", rf_grid.best_params_)
print("Best Cross-Validation ROC-AUC:", round(rf_grid.best_score_, 4))

Best Random Forest Parameters: {'classifier__max_depth': None, 'smote__k_neighbors': 5}
Best Cross-Validation ROC-AUC: 0.9505


In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score

lr_best = lr_grid.best_estimator_
rf_best = rf_grid.best_estimator_

for name, model in [("Logistic Regression", lr_best), ("Random Forest", rf_best)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print("=" * 50)
    print(f"MODEL: {name}")
    print("=" * 50)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(f"\nPrecision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

MODEL: Logistic Regression
Confusion Matrix:
[[1935   54]
 [   1   10]]

Precision: 0.1562
Recall:    0.9091
F1-Score:  0.2667
ROC-AUC:   0.9779

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99      1989
           1       0.16      0.91      0.27        11

    accuracy                           0.97      2000
   macro avg       0.58      0.94      0.63      2000
weighted avg       0.99      0.97      0.98      2000

MODEL: Random Forest
Confusion Matrix:
[[1989    0]
 [   4    7]]

Precision: 1.0000
Recall:    0.6364
F1-Score:  0.7778
ROC-AUC:   0.9683

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1989
           1       1.00      0.64      0.78        11

    accuracy                           1.00      2000
   macro avg       1.00      0.82      0.89      2000
weighted avg       1.00      1.00      1.00      2000

